# 🚀 Fine-tuning Kilo Code (générique) avec Qwen2.5-Coder-7B-Instruct

Ce notebook propose un pipeline *générique* pour fine-tuner un LLM (par défaut **Qwen2.5-Coder-7B-Instruct**) sur des jeux de données code/web **text-only**.

Fonctions clés :
- **Loaders génériques** via une classe abstraite (patron) + implémentations :
  - `RepoUrlLoader` (clonage d'un dépôt GitHub)
  - `StackSmolLoader` (dataset Hugging Face : `bigcode/the-stack-smol`)
- **Prétraitement optionnel** (`USE_TOKENIZATION`) qui convertit `(prompt, completion)` en `input_ids/labels` pour Causal LM.
- **QLoRA** (quantization 4-bit propre via `BitsAndBytesConfig`).
- Entraînement, sauvegarde et test rapide.

**Remarque licences/données :** si vous utilisez des sources externes (GitHub/HF), respectez leurs licences/conditions.


In [1]:
!pip install -q --upgrade pip
!pip install -q transformers datasets peft accelerate bitsandbytes tqdm

In [ ]:
import os, sys, json, subprocess, tempfile, shutil, random
from pathlib import Path
from typing import List, Dict
from abc import ABC, abstractmethod
from loaders.dataset_loader_base import DatasetLoaderBase
from loaders.dataset_loaders_impl import RepoUrlLoader, StackSmolLoader
from huggingface_hub import login
login(token="YOUR_HUGGINGFACE_TOKEN_HERE")


import torch
print("CUDA dispo:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(round(torch.cuda.get_device_properties(0).total_memory/1024**3,2), "GB VRAM")

CUDA dispo: False


## ⚙️ Configuration générale
- `DATA_SOURCE`: choisissez `"repo_url"` ou `"stack_smol"`.
- `USE_TOKENIZATION`: `True` conseillé pour les modèles Causal LM.
- `MAX_EXAMPLES`: nombre maximum d'exemples à générer depuis le dataset.
- `MAX_STEPS`: nombre maximum de steps d'entraînement (None = entraînement complet sur toutes les époques).
- `NUM_TRAIN_EPOCHS`: nombre d'époques si MAX_STEPS=None.
- Ajustez `MAX_FILES_PER_REPO`, etc.

In [ ]:
# 📂 Répertoires dataset
DATASET_DIR = Path("./out_dataset")
DATASET_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_FILE = DATASET_DIR / "train.jsonl"
VALID_FILE = DATASET_DIR / "valid.jsonl"
TEST_FILE  = DATASET_DIR / "test.jsonl"
STACK_SMOL_LANGS = None

# 🌐 Sélection de la source de données ("repo_url" ou "stack_smol")
DATA_SOURCE = "stack_smol"  # ⬅️ change en "stack_smol" pour utiliser the-stack-smol

# 🔗 Paramètres pour RepoUrlLoader
REPO_URL = "https://github.com/sidikfaha/nextjs-complete-starter-template"
MAX_FILES_PER_REPO = 400

# 📦 Paramètres communs dataset
MAX_EXAMPLES = 2000  # nombre max d'exemples générés
DEBUG = True
INJECT_MCP = True

# 🏃 Paramètres d'entraînement
MAX_STEPS = None  # nombre max de steps (None = entraînement complet sur toutes les époques, -1 en interne)
NUM_TRAIN_EPOCHS = 3  # nombre d'époques si MAX_STEPS=None
# 💡 Exemples d'utilisation :
# MAX_STEPS = 100  # arrêter après 100 steps (entraînement rapide pour test)
# MAX_STEPS = None; NUM_TRAIN_EPOCHS = 3  # entraînement complet sur 3 époques

# 🧪 Prétraitement (tokenization) pour Causal LM
USE_TOKENIZATION = True  # passer à False si vous avez un autre formalisme de dataset

# 🧰 Langages web à retenir pour the-stack-smol
STACK_SMOL_LANGS = ["javascript", "typescript", "html", "css"]

# 🔢 Découpage 80/10/10
SPLIT_RATIOS = (0.8, 0.1, 0.1)

random.seed(42)

## 🧱 Patron de loaders (classe abstraite) + implémentations
Chaque loader doit implémenter `fetch()` et `transform_to_examples()` pour retourner des `[{prompt, completion, ...}, ...]`.
La méthode commune `save_splits()` écrit `train/valid/test` en JSONL.

In [5]:
class DatasetLoaderBase(ABC):
    def __init__(self, out_dir: Path, inject_mcp: bool = False, debug: bool = False,
                 max_examples: int = 1000):
        self.out_dir = out_dir
        self.inject_mcp = inject_mcp
        self.debug = debug
        self.max_examples = max_examples
        self.out_dir.mkdir(parents=True, exist_ok=True)

    @abstractmethod
    def fetch(self):
        """Télécharge/charge les données brutes (git clone, HF load, etc.)."""
        pass

    @abstractmethod
    def transform_to_examples(self) -> List[Dict]:
        """Transforme les données brutes en liste de dicts {prompt, completion, ...}."""
        pass

    def save_splits(self, examples: List[Dict], ratios=(0.8, 0.1, 0.1)):
        n = len(examples)
        if self.debug:
            print(f"[DEBUG] Total exemples préparés: {n}")
        if n == 0:
            # fichiers vides pour ne pas bloquer le pipeline
            for split in ["train.jsonl", "valid.jsonl", "test.jsonl"]:
                (self.out_dir / split).write_text("")
            return

        # shuffle pour un split stable
        exs = examples[:]
        random.shuffle(exs)
        r_train, r_valid, r_test = ratios
        n_train = int(n * r_train)
        n_valid = int(n * (r_train + r_valid))
        splits = {
            "train": exs[:n_train],
            "valid": exs[n_train:n_valid],
            "test": exs[n_valid:]
        }
        import json
        for split, arr in splits.items():
            with open(self.out_dir / f"{split}.jsonl", "w", encoding="utf-8") as f:
                for ex in arr:
                    f.write(json.dumps(ex, ensure_ascii=False) + "\n")
        if self.debug:
            print(f"[DEBUG] Sauvegardé → train={len(splits['train'])}, valid={len(splits['valid'])}, test={len(splits['test'])}")


class RepoUrlLoader(DatasetLoaderBase):
    VALID_EXT = [".js", ".jsx", ".ts", ".tsx", ".json", ".md", ".html", ".css"]
    def __init__(self, repo_url: str, out_dir: Path, inject_mcp=False, debug=False,
                 max_files=400, max_examples=1000):
        super().__init__(out_dir, inject_mcp, debug, max_examples)
        self.repo_url = repo_url
        self.max_files = max_files
        self.local_path = None

    def fetch(self):
        tmp = tempfile.mkdtemp()
        if self.debug:
            print(f"[DEBUG] git clone --depth 1 {self.repo_url} → {tmp}")
        subprocess.run(["git", "clone", "--depth", "1", self.repo_url, tmp], check=True)
        self.local_path = Path(tmp)

    def transform_to_examples(self) -> List[Dict]:
        files = []
        for root, _, fnames in os.walk(self.local_path):
            for fname in fnames:
                if any(fname.endswith(ext) for ext in self.VALID_EXT):
                    fpath = Path(root) / fname
                    try:
                        if fpath.stat().st_size < 300_000:  # évite gros fichiers
                            files.append(fpath)
                    except Exception:
                        continue
        files = files[: self.max_files]
        examples = []
        for f in files:
            try:
                text = f.read_text(encoding="utf-8", errors="ignore")
            except Exception as e:
                if self.debug:
                    print(f"[DEBUG] Lecture impossible {f}: {e}")
                continue
            obj = {
                "prompt": f"Contenu du fichier {f.name} : explique, améliore ou complète.",
                "completion": text
            }
            if self.inject_mcp:
                obj["mcp"] = {"tool": "kilo-code", "version": "1.0"}
            examples.append(obj)
            if len(examples) >= self.max_examples:
                break
        return examples


class StackSmolLoader(DatasetLoaderBase):
    def __init__(self, out_dir: Path, inject_mcp=False, debug=False,
                 keep_langs=STACK_SMOL_LANGS, max_examples=1000):
        super().__init__(out_dir, inject_mcp, debug, max_examples)
        self.keep_langs = keep_langs
        self.ds = None
        self.local_dataset_dir = Path("./datasets/the-stack-smol")

    def fetch(self):
        from datasets import load_dataset
        import requests
        
        # Vérifier si le cache local existe
        if self.local_dataset_dir.exists():
            print("📂 Dataset trouvé en local →", self.local_dataset_dir)
            # Charger depuis le cache local
            from datasets import load_from_disk
            self.ds = load_from_disk(str(self.local_dataset_dir))
        else:
            print("🌐 Vérification de la taille du dataset the-stack-smol...")
            
            # Vérifier la taille du dataset sur Hugging Face
            try:
                response = requests.get("https://huggingface.co/api/datasets/bigcode/the-stack-smol", timeout=10)
                if response.status_code == 200:
                    dataset_info = response.json()
                    # Calculer la taille approximative (en Go)
                    size_gb = sum(split.get("size", 0) for split in dataset_info.get("splits", {}).values()) / (1024**3)
                    print(f"📊 Taille estimée du dataset: {size_gb:.2f} Go")
                    
                    if size_gb > 5.0:
                        print("🚀 Dataset > 5Go, utilisation du mode streaming...")
                        self.ds = load_dataset("bigcode/the-stack-smol", streaming=True)["train"]
                        self.streaming_mode = True
                    else:
                        print("💾 Dataset ≤ 5Go, téléchargement complet...")
                        os.makedirs("./datasets", exist_ok=True)
                        self.ds = load_dataset("bigcode/the-stack-smol", cache_dir=str(self.local_dataset_dir))["train"]
                        # Sauvegarder localement pour les futures utilisations
                        self.ds.save_to_disk(str(self.local_dataset_dir))
                        print("✅ Dataset téléchargé et sauvegardé dans", self.local_dataset_dir)
                        self.streaming_mode = False
                else:
                    raise Exception("Impossible de récupérer les infos du dataset")
            except Exception as e:
                print(f"⚠️ Impossible de vérifier la taille ({e}), utilisation du mode normal...")
                os.makedirs("./datasets", exist_ok=True)
                self.ds = load_dataset("bigcode/the-stack-smol", cache_dir=str(self.local_dataset_dir))["train"]
                self.ds.save_to_disk(str(self.local_dataset_dir))
                self.streaming_mode = False
        
        # Appliquer les filtres de langues (uniquement si pas en streaming)
        if self.keep_langs and not getattr(self, 'streaming_mode', False):
            keep_langs_lower = [lang.lower() for lang in self.keep_langs]
            self.ds = self.ds.filter(lambda ex: ex.get("lang", "").lower() in keep_langs_lower)
        elif self.keep_langs and getattr(self, 'streaming_mode', False):
            print("ℹ️ Mode streaming: filtrage des langues appliqué lors de la transformation")
        
        if self.debug:
            print(self.ds)

    def transform_to_examples(self) -> List[Dict]:
        examples = []
        if self.ds is None:
            return examples
        
        # Préparer le filtrage des langues pour le mode streaming
        keep_langs_lower = [lang.lower() for lang in self.keep_langs] if self.keep_langs else None
        
        # stream/iterate
        for ex in self.ds:
            # Appliquer le filtrage des langues en mode streaming
            if keep_langs_lower and getattr(self, 'streaming_mode', False):
                lang = ex.get("lang", "").lower()
                if lang not in keep_langs_lower:
                    continue
            
            prompt = f"Fichier {ex.get('lang','code')} : analyse, explique ou complète."
            completion = ex.get("content", "")
            obj = {"prompt": prompt, "completion": completion}
            if self.inject_mcp:
                obj["mcp"] = {"tool": "kilo-code", "version": "1.0"}
            examples.append(obj)
            if len(examples) >= self.max_examples:
                break
        return examples


## 📥 Génération du dataset (si `train.jsonl` absent)
Choix automatique du loader selon `DATA_SOURCE`.

In [6]:
if not TRAIN_FILE.exists():
    print("📥 Dataset absent, génération en cours…")
    if DATA_SOURCE == "repo_url":
        loader = RepoUrlLoader(
            repo_url=REPO_URL,
            out_dir=DATASET_DIR,
            inject_mcp=INJECT_MCP,
            debug=DEBUG,
            max_files=MAX_FILES_PER_REPO,
            max_examples=MAX_EXAMPLES,
        )
    elif DATA_SOURCE == "stack_smol":
        loader = StackSmolLoader(
            out_dir=DATASET_DIR,
            inject_mcp=INJECT_MCP,
            debug=DEBUG,
            keep_langs=STACK_SMOL_LANGS,
            max_examples=MAX_EXAMPLES,
        )
    else:
        raise ValueError("DATA_SOURCE invalide (choisir 'repo_url' ou 'stack_smol')")

    try:
        loader.fetch()
        examples = loader.transform_to_examples()
        loader.save_splits(examples, ratios=SPLIT_RATIOS)
        print("✅ Dataset généré →", DATASET_DIR)
    except Exception as e:
        print("⚠️ Erreur génération dataset:", e)
        for f in [TRAIN_FILE, VALID_FILE, TEST_FILE]:
            f.write_text("")
else:
    print("✅ Dataset déjà présent, on passe directement au fine-tuning.")

✅ Dataset déjà présent, on passe directement au fine-tuning.


## 📦 Chargement du dataset JSONL

In [7]:
from datasets import load_dataset
dataset = load_dataset("json", data_files={
    "train": str(TRAIN_FILE),
    "validation": str(VALID_FILE),
    "test": str(TEST_FILE)
})
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion', 'mcp'],
        num_rows: 1600
    })
    validation: Dataset({
        features: ['prompt', 'completion', 'mcp'],
        num_rows: 200
    })
    test: Dataset({
        features: ['prompt', 'completion', 'mcp'],
        num_rows: 200
    })
})

## 🧠 Chargement du modèle (QLoRA 4-bit propre)
Utilise `BitsAndBytesConfig` (recommandé) au lieu de l'argument `load_in_4bit` déprécié.

**⚠️ Problème de mémoire détecté :** Si vous rencontrez l'erreur "fichier de pagination insuffisant", utilisez un modèle plus petit :

```python
# Remplacez cette ligne dans la cellule suivante :
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"  # 7B paramètres

# Par l'une de ces alternatives :
MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"   # 3B paramètres (recommandé)
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct" # 1.5B paramètres (très léger)
```

Les modèles plus petits nécessitent moins de RAM et fonctionnent mieux sur Windows.

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"  # Réf. HF
LOCAL_MODEL_DIR = "./models/Qwen2.5-Coder-7B-Instruct"

# Forcer le téléchargement et stockage local du modèle
if not os.path.exists(LOCAL_MODEL_DIR):
    print("🌐 Téléchargement du modèle depuis Hugging Face vers le dossier local...")
    os.makedirs("./models", exist_ok=True)
    
    # Télécharger le tokenizer
    tokenizer_temp = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer_temp.save_pretrained(LOCAL_MODEL_DIR)
    
    # Télécharger le modèle (sans quantization pour le stockage)
    model_temp = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    model_temp.save_pretrained(LOCAL_MODEL_DIR)
    
    print("✅ Modèle téléchargé et sauvegardé dans", LOCAL_MODEL_DIR)
else:
    print("📂 Modèle trouvé en local →", LOCAL_MODEL_DIR)

# Utiliser toujours le modèle local
model_path = LOCAL_MODEL_DIR

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# Stratégie de chargement avec fallbacks pour gérer les contraintes mémoire Windows
try:
    print("🔄 Tentative de chargement en QLoRA 4-bit (recommandé)...")
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        trust_remote_code=True,
        quantization_config=bnb_config,
    )
    print("✅ Modèle chargé en QLoRA 4-bit")
except Exception as e1:
    print(f"⚠️ Échec QLoRA 4-bit: {str(e1)[:100]}...")
    try:
        print("🔄 Tentative de chargement FP16 avec limites mémoire...")
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            device_map={"": 0},
            max_memory={0: "4GB", "cpu": "12GB"},
            offload_folder="E:/temp_offload",
            trust_remote_code=True,
            torch_dtype=torch.float16,
        )
        print("✅ Modèle chargé en FP16 avec limites mémoire")
    except Exception as e2:
        print(f"⚠️ Échec FP16: {str(e2)[:100]}...")
        try:
            print("🔄 Tentative de chargement CPU-only...")
            model = AutoModelForCausalLM.from_pretrained(
                model_path,
                device_map={"": "cpu"},
                trust_remote_code=True,
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
            )
            print("✅ Modèle chargé en CPU-only")
        except Exception as e3:
            print(f"⚠️ Échec CPU-only: {str(e3)[:100]}...")
            try:
                print("🔄 Dernière tentative en 8-bit...")
                model = AutoModelForCausalLM.from_pretrained(
                    model_path,
                    device_map="auto",
                    trust_remote_code=True,
                    load_in_8bit=True,
                    low_cpu_mem_usage=True,
                )
                print("✅ Modèle chargé en 8-bit (configuration minimale)")
            except Exception as e4:
                print(f"❌ Toutes les méthodes de chargement ont échoué.")
                print("💡 Solutions alternatives :")
                print("   1. Utiliser un modèle plus petit (ex: Qwen2.5-3B ou 1.5B)")
                print("   2. Augmenter la RAM physique de votre système")
                print("   3. Fermer les autres applications utilisant de la mémoire")
                print("   4. Utiliser un modèle déjà quantisé depuis Hugging Face")
                raise RuntimeError("Impossible de charger le modèle avec la configuration actuelle")

print("✅ Modèle prêt")

📂 Modèle trouvé en local → ./models/Qwen2.5-Coder-7B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


🔄 Tentative de chargement en QLoRA 4-bit (recommandé)...
⚠️ Échec QLoRA 4-bit: cannot import name 'get_cached_models' from 'transformers.utils' (e:\kilo_code_finetune_qwen_package...
🔄 Tentative de chargement FP16 avec limites mémoire...
⚠️ Échec FP16: cannot import name 'get_cached_models' from 'transformers.utils' (e:\kilo_code_finetune_qwen_package...
🔄 Tentative de chargement CPU-only...
⚠️ Échec CPU-only: cannot import name 'get_cached_models' from 'transformers.utils' (e:\kilo_code_finetune_qwen_package...
🔄 Dernière tentative en 8-bit...
❌ Toutes les méthodes de chargement ont échoué.
💡 Solutions alternatives :
   1. Utiliser un modèle plus petit (ex: Qwen2.5-3B ou 1.5B)
   2. Augmenter la RAM physique de votre système
   3. Fermer les autres applications utilisant de la mémoire
   4. Utiliser un modèle déjà quantisé depuis Hugging Face


RuntimeError: Impossible de charger le modèle avec la configuration actuelle

## 🧪 Prétraitement optionnel (tokenization)
Activez/désactivez via `USE_TOKENIZATION`.

In [9]:
if USE_TOKENIZATION:
    def tokenize_function(examples):
        texts = [ (p or "") + "\n" + (c or "") for p, c in zip(examples.get("prompt", [""]*len(examples["completion"])),
                                                                  examples.get("completion", [""]))]
        model_inputs = tokenizer(texts, max_length=256, truncation=True, padding=True)
        model_inputs["labels"] = model_inputs["input_ids"].copy()
        return model_inputs

    tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset["train"].column_names)
else:
    tokenized_dataset = dataset  # laisser brut si vous avez un autre collator/custom forward

tokenized_dataset

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1600
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
})

## 🪡 Appliquer LoRA (PEFT)

In [10]:
from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj","v_proj"],
    bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,523,136 || all params: 7,618,139,648 || trainable%: 0.0331


## 🏃 Entraînement (Trainer)
Note : `remove_unused_columns=False` est important quand on alimente directement `input_ids/labels`.

In [13]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    max_steps=MAX_STEPS if MAX_STEPS is not None else -1,  # -1 = entraînement complet
    num_train_epochs=NUM_TRAIN_EPOCHS if MAX_STEPS is None else 1,  # époques si pas de max_steps
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    eval_strategy="epoch",
    per_device_eval_batch_size=1,
    remove_unused_columns=False,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.875000,0.943426
2,0.836000,0.936196
3,0.817600,0.935661


TrainOutput(global_step=600, training_loss=0.893580900033315, metrics={'train_runtime': 4155.616, 'train_samples_per_second': 1.155, 'train_steps_per_second': 0.144, 'total_flos': 5.21488633430016e+16, 'train_loss': 0.893580900033315, 'epoch': 3.0})

## 💾 Sauvegarde de l'adaptateur LoRA + tokenizer

In [14]:
OUTPUT_DIR = "./kilo_code_qwen_lora"
Path(OUTPUT_DIR).mkdir(exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✅ Sauvegardé dans", OUTPUT_DIR)

✅ Sauvegardé dans ./kilo_code_qwen_lora


## 🧪 Test rapide (génération)
Augmentez `max_new_tokens` pour des snippets plus longs.

In [15]:
from transformers import pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto")
res = pipe("Crée une page Next.js avec un formulaire de contact", max_new_tokens=300, temperature=0.7, top_p=0.9)
print(res[0]['generated_text'])

Device set to use cuda:0


Crée une page Next.js avec un formulaire de contact qui envoie les informations à un webhook. Le formulaire doit avoir des champs pour le nom, l'adresse email et le message. Après soumission du formulaire, affichez un message de confirmation.

Voici ce que vous avez déjà fait :

1. Créez une nouvelle application Next.js.
2. Installez axios pour les requêtes HTTP.
3. Ajoutez le code suivant à votre page index.js.

```
import Head from 'next/head'
import Image from 'next/image'
import styles from '../styles/Home.module.css'

export default function Home() {
  return (
    <div className={styles.container}>
      <Head>
        <title>Contact</title>
        <meta name="description" content="Generated by create next app" />
        <link rel="icon" href="/favicon.ico" />
      </Head>

      <main className={styles.main}>
        <h1 className={styles.title}>
          Contact
        </h1>

        <form action="#" method="post">
          <label htmlFor="">Name: </label>
          <inpu

In [1]:
# =====================================================================
# 🚀 Conversion d'un modèle HF (fine-tuné) vers GGUF 4-bit (Q4_K_M)
# Compatible LM Studio, llamafile, ollama, koboldcpp, etc.
# =====================================================================

# 💡 Conseils mémoire pour Windows :
# - Le modèle Qwen2.5-7B nécessite ~14GB VRAM en FP16
# - Si vous avez < 14GB VRAM, le code utilise automatiquement l'offloading
# - En cas d'erreur mémoire, le fallback 8-bit sera utilisé automatiquement
# - L'offloading utilise maintenant E:/temp_offload pour éviter les problèmes d'espace sur C:

import os
import subprocess
import torch

# 📂 Chemin vers ton modèle fine-tuné Hugging Face/PEFT
HF_MODEL_PATH = "./kilo_code_qwen_lora"   # <-- adapte si besoin

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"  # Réf. HF
LOCAL_MODEL_DIR = "./models/Qwen2.5-Coder-7B-Instruct"

if os.path.exists(LOCAL_MODEL_DIR):
    print("📂 Modèle de base trouvé en local →", LOCAL_MODEL_DIR)
    base_model_path = LOCAL_MODEL_DIR
else:
    print("🌐 Téléchargement du modèle de base depuis Hugging Face…")
    base_model_path = MODEL_NAME

# 📂 Chemin de sortie GGUF
GGUF_OUT_DIR = "./gguf_export"
os.makedirs(GGUF_OUT_DIR, exist_ok=True)

# 🏷️ Nom du modèle converti
GGUF_MODEL_NAME = "qwen2.5-coder-7b-kilo-finetuned-q6k.gguf"

# =====================================================================
# ⚙️ Étape préalable : fusion du modèle LoRA avec le modèle de base
# =====================================================================

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("🔄 Fusion du modèle LoRA avec le modèle de base...")

# Charger le tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

# Charger le modèle de base
print("🔄 Chargement du modèle de base (mémoire optimisée)...")

# Créer le dossier d'offload sur E: s'il n'existe pas
os.makedirs("E:/temp_offload", exist_ok=True)

try:
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_path,  # Utiliser le chemin local si disponible
        torch_dtype=torch.float16,
        device_map="auto",  # Utiliser CPU et GPU pour accélérer le chargement
        low_cpu_mem_usage=True,
        # Optimisations mémoire pour Windows
        max_memory={0: "4GB", "cpu": "12GB"},  # Réduire GPU, augmenter CPU pour éviter la pagination
        offload_folder="E:/temp_offload",  # Dossier sur E: pour l'offloading (plus d'espace)
    )
    print("✅ Modèle chargé avec succès")
except Exception as e:
    print(f"⚠️ Échec du chargement standard ({e})")
    print("🔄 Tentative avec configuration ultra-conservative...")
    try:
        # Configuration ultra-conservative : tout sur CPU avec offloading
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_path,
            torch_dtype=torch.float16,
            device_map={"": "cpu"},  # Forcer tout sur CPU
            low_cpu_mem_usage=True,
            offload_folder="E:/temp_offload",
            max_memory={"cpu": "16GB"},  # Maximum sur CPU
        )
        print("✅ Modèle chargé en mode CPU-only")
    except Exception as e2:
        print(f"⚠️ Échec du mode CPU-only ({e2})")
        print("🔄 Dernière tentative avec configuration 8-bit...")
        # Fallback en 8-bit si pas assez de mémoire
        from transformers import BitsAndBytesConfig
        bnb_config_8bit = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_compute_dtype=torch.float16,
        )
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_path,
            quantization_config=bnb_config_8bit,
            device_map="auto",
            low_cpu_mem_usage=True,
            max_memory={0: "2GB", "cpu": "10GB"},  # Configuration minimale pour 8-bit
            offload_folder="E:/temp_offload",
        )
        print("✅ Modèle chargé en 8-bit (configuration minimale)")
    except Exception as e3:
        print(f"❌ Toutes les méthodes de chargement ont échoué.")
        print("💡 Solutions alternatives :")
        print("   1. Utiliser un modèle plus petit (ex: Qwen2.5-3B ou 1.5B)")
        print("   2. Augmenter la RAM physique de votre système")
        print("   3. Fermer les autres applications utilisant de la mémoire")
        print("   4. Utiliser un modèle déjà quantisé depuis Hugging Face")
        raise RuntimeError("Impossible de charger le modèle avec la configuration actuelle")

# Charger l'adaptateur LoRA
print(f"🔄 Chargement de l'adaptateur LoRA depuis {HF_MODEL_PATH}...")

# Vérifier d'abord la compatibilité de l'adaptateur
try:
    from peft import PeftConfig
    peft_config = PeftConfig.from_pretrained(HF_MODEL_PATH)
    print(f"📋 Configuration LoRA trouvée: {peft_config.base_model_name_or_path}")

    # Vérifier si le modèle de base correspond
    if peft_config.base_model_name_or_path != base_model_path and peft_config.base_model_name_or_path != MODEL_NAME:
        print(f"⚠️ Incompatibilité détectée:")
        print(f"   Adaptateur entraîné sur: {peft_config.base_model_name_or_path}")
        print(f"   Modèle de base actuel: {base_model_path}")
        print("🔄 Tentative de chargement malgré l'incompatibilité...")

except Exception as config_error:
    print(f"⚠️ Impossible de lire la configuration LoRA: {config_error}")

try:
    model = PeftModel.from_pretrained(base_model, HF_MODEL_PATH)
    print("✅ Adaptateur LoRA chargé avec succès")
except KeyError as e:
    print(f"⚠️ Erreur de compatibilité LoRA: {e}")
    print("💡 Causes possibles :")
    print("   - L'adaptateur LoRA a été entraîné sur un modèle différent")
    print("   - Incompatibilité de version entre le modèle de base et l'adaptateur")
    print("   - Problème de sauvegarde/chargement de l'adaptateur")
    print()
    print("🔄 Tentatives de récupération...")

    # Tentative 1: Recharger le modèle fine-tuné directement (si sauvegardé avec merge)
    try:
        print("🔄 Tentative 1: Rechargement direct du modèle fine-tuné...")
        merged_model = AutoModelForCausalLM.from_pretrained(
            HF_MODEL_PATH,
            torch_dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True,
            max_memory={0: "4GB", "cpu": "12GB"},
            offload_folder="E:/temp_offload",
        )
        print("✅ Modèle fine-tuné rechargé directement (fusion déjà faite)")
        MERGED_MODEL_PATH = HF_MODEL_PATH  # Utiliser le dossier existant
    except Exception as e2:
        print(f"⚠️ Échec rechargement direct: {e2}")

        # Tentative 2: Utiliser le modèle de base seul
        print("🔄 Tentative 2: Utilisation du modèle de base non fine-tuné...")
        merged_model = base_model
        MERGED_MODEL_PATH = "./base_model_only"
        merged_model.save_pretrained(MERGED_MODEL_PATH)
        tokenizer.save_pretrained(MERGED_MODEL_PATH)
        print("✅ Modèle de base sauvegardé (sans fine-tuning)")
        print("ℹ️ Note: Le modèle GGUF final n'aura pas le fine-tuning appliqué")
        print("❌ ERREUR: Impossible de fusionner le LoRA. Le GGUF sera du modèle de base non fine-tuné.")
        print("💡 Solution: Vérifiez la compatibilité de votre adaptateur LoRA avec le modèle de base.")
        raise RuntimeError("Fusion LoRA échouée - modèle de base utilisé à la place. Conversion GGUF annulée.")

else:
    # Si le chargement LoRA a réussi, procéder à la fusion
    print("🔄 Fusion de l'adaptateur LoRA avec le modèle de base...")
    merged_model = model.merge_and_unload()
    print("✅ Modèle fusionné avec succès")

    # Sauvegarder le modèle fusionné
    MERGED_MODEL_PATH = "./merged_model"
    merged_model.save_pretrained(MERGED_MODEL_PATH, max_shard_size="1GB")
    tokenizer.save_pretrained(MERGED_MODEL_PATH)
    print("✅ Modèle fusionné sauvegardé dans", MERGED_MODEL_PATH)

# =====================================================================
# ⚙️ Conversion via llama.cpp (fichiers binaires nécessaires)
# Tu dois avoir cloné et compilé llama.cpp :
# git clone https://github.com/ggerganov/llama.cpp
# cd llama.cpp && mkdir build && cd build && cmake .. && make -j
# =====================================================================

LLAMA_CPP_CONVERT = "./llama.cpp/convert_hf_to_gguf.py"
LLAMA_CPP_QUANTIZE = "./llama.cpp/build/bin/Release/llama-quantize.exe"

# Étape 1 : conversion du modèle HF → format f16 (intermédiaire)
f16_path = os.path.join(GGUF_OUT_DIR, "model-f16.gguf")
if os.path.exists(f16_path):
    print(f"📂 Fichier intermédiaire trouvé: {f16_path} - Conversion ignorée")
else:
    cmd_convert = [
        "python", LLAMA_CPP_CONVERT,
        MERGED_MODEL_PATH,
        "--outfile", f16_path,
        "--outtype", "f16"
    ]
    print("🔄 Conversion HF → GGUF f16...")
    subprocess.run(cmd_convert, check=True)

# Étape 2 : quantization f16 → Q6_K
cmd_quant = [
    LLAMA_CPP_QUANTIZE,
    os.path.join(GGUF_OUT_DIR, "model-f16.gguf"),
    os.path.join(GGUF_OUT_DIR, GGUF_MODEL_NAME),
    "Q6_K"
]
print("🔄 Quantization vers Q6_K...")
subprocess.run(cmd_quant, check=True)

print(f"✅ Conversion terminée → {os.path.join(GGUF_OUT_DIR, GGUF_MODEL_NAME)}")

# 🧹 Nettoyage optionnel du dossier d'offload
import shutil
offload_dir = "E:/temp_offload"
if os.path.exists(offload_dir):
    try:
        shutil.rmtree(offload_dir)
        print(f"🧹 Dossier d'offload nettoyé: {offload_dir}")
    except Exception as e:
        print(f"⚠️ Impossible de nettoyer {offload_dir}: {e}")

📂 Modèle de base trouvé en local → ./models/Qwen2.5-Coder-7B-Instruct
🔄 Fusion du modèle LoRA avec le modèle de base...
🔄 Fusion du modèle LoRA avec le modèle de base...


Device 0 is not available, available devices are []


🔄 Chargement du modèle de base (mémoire optimisée)...
⚠️ Échec du chargement standard (Cannot access accelerator device when none is available.)
🔄 Tentative avec configuration ultra-conservative...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The 8-bit optimizer is not available on your device, only available on CUDA for now.


✅ Modèle chargé en mode CPU-only
🔄 Chargement de l'adaptateur LoRA depuis ./kilo_code_qwen_lora...
📋 Configuration LoRA trouvée: ./models/Qwen2.5-Coder-7B-Instruct
✅ Adaptateur LoRA chargé avec succès
🔄 Fusion de l'adaptateur LoRA avec le modèle de base...
✅ Adaptateur LoRA chargé avec succès
🔄 Fusion de l'adaptateur LoRA avec le modèle de base...
✅ Modèle fusionné avec succès
✅ Modèle fusionné avec succès
✅ Modèle fusionné sauvegardé dans ./merged_model
📂 Fichier intermédiaire trouvé: ./gguf_export\model-f16.gguf - Conversion ignorée
🔄 Quantization vers Q6_K...
✅ Modèle fusionné sauvegardé dans ./merged_model
📂 Fichier intermédiaire trouvé: ./gguf_export\model-f16.gguf - Conversion ignorée
🔄 Quantization vers Q6_K...
✅ Conversion terminée → ./gguf_export\qwen2.5-coder-7b-kilo-finetuned-q6k.gguf
🧹 Dossier d'offload nettoyé: E:/temp_offload
✅ Conversion terminée → ./gguf_export\qwen2.5-coder-7b-kilo-finetuned-q6k.gguf
🧹 Dossier d'offload nettoyé: E:/temp_offload
